In [1]:
from google.colab import files
uploaded = files.upload()

Saving orders_cleaned (1).csv to orders_cleaned (1).csv


In [2]:
!pip install pyspark==3.5.3 delta-spark==3.2.0 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 13.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.3 which is incompatible.


In [3]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import col, when
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("CustomerOrderInsightsETL") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [5]:
df = spark.read.csv("/content/orders_cleaned (1).csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- delivery_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- issue: string (nullable = true)
 |-- is_delivered: boolean (nullable = true)
 |-- delay_days: integer (nullable = true)
 |-- delayed: integer (nullable = true)

+--------+-----------+-------------+------+----------+-------------+------+--------------------+------------+----------+-------+
|order_id|customer_id|customer_name|region|order_date|delivery_date|amount|               issue|is_delivered|delay_days|delayed|
+--------+-----------+-------------+------+----------+-------------+------+--------------------+------------+----------+-------+
|       1|          1|   Asha Patel|  West|2026-06-01|   2026-06-08|1500.0|       Courier delay|        true|         2|      1|
|       2|          2|  

In [6]:
new_updates = spark.createDataFrame([
    Row(order_id=3, delivery_date="2026-06-24", is_delivered=True),
    Row(order_id=19, delivery_date="2026-06-30", is_delivered=True),
], ["order_id", "delivery_date", "is_delivered"])

new_updates.show()

+--------+-------------+------------+
|order_id|delivery_date|is_delivered|
+--------+-------------+------------+
|       3|   2026-06-24|        true|
|      19|   2026-06-30|        true|
+--------+-------------+------------+



In [7]:
updates_renamed = new_updates \
    .withColumnRenamed("delivery_date", "new_delivery_date") \
    .withColumnRenamed("is_delivered", "new_is_delivered")

updated_df = df.join(updates_renamed, on="order_id", how="left")

updated_df = updated_df.withColumn(
    "delivery_date",
    when(col("new_delivery_date").isNotNull(), col("new_delivery_date")).otherwise(col("delivery_date"))
).withColumn(
    "is_delivered",
    when(col("new_is_delivered").isNotNull(), col("new_is_delivered")).otherwise(col("is_delivered"))
).drop("new_delivery_date", "new_is_delivered")

# Check that orders 3 and 19 were updated
updated_df.filter(col("order_id").isin(3, 19)).select(
    "order_id", "delivery_date", "is_delivered"
).show()

+--------+-------------+------------+
|order_id|delivery_date|is_delivered|
+--------+-------------+------------+
|      19|   2026-06-30|        true|
|       3|   2026-06-24|        true|
+--------+-------------+------------+



In [8]:
updated_df.write.format("delta").mode("overwrite").save("delta/orders_updated")

updated_df.coalesce(1).write.mode("overwrite").option("header", True) \
    .csv("orders_updated_csv")

print("Saved as Delta table (delta/orders_updated/) and as CSV (orders_updated_csv/)")

Saved as Delta table (delta/orders_updated/) and as CSV (orders_updated_csv/)


In [9]:
delta_df = spark.read.format("delta").load("delta/orders_updated")
delta_df.createOrReplaceTempView("orders_updated")

top_5_delayed = spark.sql("""
    SELECT customer_id, customer_name, SUM(delay_days) AS total_delay_days
    FROM orders_updated
    GROUP BY customer_id, customer_name
    ORDER BY total_delay_days DESC
    LIMIT 5
""")

top_5_delayed.show()

+-----------+-------------+----------------+
|customer_id|customer_name|total_delay_days|
+-----------+-------------+----------------+
|          4|  John Carter|               7|
|          1|   Asha Patel|               6|
|          3|   Meena Iyer|               4|
|         10|   Karan Shah|               1|
|          6| Vikram Singh|               1|
+-----------+-------------+----------------+



In [10]:
import glob, shutil
part_file = glob.glob("orders_updated_csv/part-*.csv")[0]
shutil.copy(part_file, "orders_updated.csv")
files.download("orders_updated.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>